In [1]:
import json

import pandas as pd

df_chunks = pd.read_parquet("data/law_content_chunks.parquet", engine="pyarrow")

df_chunks.head(3)

,law_id,url,content_html,chunks,error
0,14/2022/TT-NHNN,https://thuvienphapluat.vn/van-ban/Tien-te-Nga...,"\r\n<!DOCTYPE HTML PUBLIC ""-//W3C//DTD HTML 4....",[{'content': 'NGÂN HÀNG NHÀ NƯỚC VIỆT NAM -...,None
1,11/2022/TT-NHNN,https://thuvienphapluat.vn/van-ban/Tien-te-Nga...,"\r\n<!DOCTYPE HTML PUBLIC ""-//W3C//DTD HTML 4....",[{'content': 'NGÂN HÀNG NHÀ NƯỚC VIỆT NAM -...,None
2,02/2007/TT-BNV,https://thuvienphapluat.vn/van-ban/Lao-dong-Ti...,"\r\n<!DOCTYPE HTML PUBLIC ""-//W3C//DTD HTML 4....",[{'content': 'BỘ NỘI VỤ ****** CỘNG HOÀ ...,None


In [3]:
df = pd.read_parquet("data/train/legal_corpus.parquet", engine="pyarrow")
df.head(3)

,id,law_id,aid,content_Article
0,0,14/2022/TT-NHNN,0,"1. Thông tư này quy định mã số, tiêu chuẩn chu..."
1,0,14/2022/TT-NHNN,1,1. Kiểm soát viên cao cấp ngân hàng Mã số: 07....
2,0,14/2022/TT-NHNN,2,"1. Có bản lĩnh chính trị vững vàng, kiên định ..."


In [25]:
from src.preprocess.text_utils import normalize_text

id = 574
law_id = df[df["aid"]==id].reset_index(drop=True)["law_id"][0]
df_temp = df_chunks[df_chunks["law_id"]==law_id].reset_index(drop=True)
url = df_temp["url"][0]
chunks = df_temp["chunks"][0]
content_html = df_temp["content_html"][0]
print("aid:", id)
print("law_id:", law_id)
print("law url:", url)
print("-"*50)
print("Nội dung của aid:\n", df["content_Article"][id])
print("-"*50)
print("Nội dung của chunks trích xuất được từ HTML:\n", chunks)
# print("-"*50)
# print(content_html)

aid: 574
law_id: 38/2015/TT-NHNN
law url: https://thuvienphapluat.vn/van-ban/Thuong-mai/Thong-tu-38-2015-TT-NHNN-sua-doi-Thong-tu-16-2012-TT-NHNN-huong-dan-hoat-dong-kinh-doanh-vang-300421.aspx
--------------------------------------------------
Nội dung của aid:
 Bãi bỏ khoản 1 Điều 4 Thông tư 16/2012/TT-NHNN .
--------------------------------------------------
Nội dung của chunks trích xuất được từ HTML:
 [{'content': 'NGÂN HÀNG NHÀ\r\n  NƯỚC VIỆT NAM ------- CỘNG HÒA XÃ HỘI\r\n  CHỦ NGHĨA VIỆT NAM Độc lập - Tự do - Hạnh phúc --------------- Số:\r\n  38/2015/TT-NHNN Hà Nội, ngày 31 tháng 12 năm 2015', 'titles': array([], dtype=object)}
 {'content': 'Căn cứ Luật Ngân\r\nhàng Nhà nước Việt Nam số 46/2010/QH12 ngày 16 tháng 6 năm 2010;\nCăn cứ Luật các tổ\r\nchức tín dụng số 47/2010/QH12 ngày 16 tháng 6 năm 2010;\nCăn cứ Nghị định số 156/2013/NĐ-CP ngày 11 tháng 11 năm 2013 của Chính phủ quy định chức năng, nhiệm vụ, quyền hạn\r\nvà cơ cấu tổ chức của Ngân hàng Nhà nước Việt Nam;\nCăn cứ

In [15]:
from src.preprocess.text_utils import is_text_similar, normalize_titles
count_miss_title = 0
df["titles"] = None

for i in range(len(df)):
    law_id = df["law_id"][i]
    df_temp = df_chunks[df_chunks["law_id"] == law_id].reset_index(drop=True)
    if len(df_temp)==0 or len(df_temp)>1:
        print(f'Không tìm thấy law_id: {law_id}')
        continue
    chunks = df_temp["chunks"][0]
    if len(chunks)==0:
        print(f'Không có chunks trong law_id: {law_id}')
        continue
    content = df["content_Article"][i]
    is_matched = False
    for chunk in chunks:
        if is_text_similar(content, chunk["content"]):
            df.at[i, "titles"] = normalize_titles(chunk["titles"])
            is_matched = True
            break

    if not is_matched:
        print(df["aid"][i])
        if len(chunks) > 1:
            df.at[i, "titles"] = normalize_titles(chunks[1]["titles"][0:1])
        else:
            df.at[i, "titles"] = []
        count_miss_title += 1
        # break

441
574
763
780
781
783
784
811
1050
1125
1127
1128
1134
1135
1175
1176
1177
1178
1179
1180
1181
1182
1183
1184
1185
1186
1187
1188
1189
1190
1191
1192
1193
1194
1195
1196
1197
1198
1199
1200
1201
1202
1203
1204
1205
1206
1207
1208
1209
1210
1211
1212
1213
1214
1215
1216
1217
1218
1219
1220
1221
1222
1223
1224
1225
1226
1227
1230
1353
1355
1357
1506
Không có chunks trong law_id: 02/2020/TT-NHNN
Không có chunks trong law_id: 02/2020/TT-NHNN
Không có chunks trong law_id: 02/2020/TT-NHNN
Không có chunks trong law_id: 02/2020/TT-NHNN
Không có chunks trong law_id: 02/2020/TT-NHNN
Không có chunks trong law_id: 02/2020/TT-NHNN
Không có chunks trong law_id: 02/2020/TT-NHNN
2364
Không có chunks trong law_id: 130/2003/QĐ-TTg
Không có chunks trong law_id: 130/2003/QĐ-TTg
Không có chunks trong law_id: 130/2003/QĐ-TTg
Không có chunks trong law_id: 130/2003/QĐ-TTg
Không có chunks trong law_id: 130/2003/QĐ-TTg
Không có chunks trong law_id: 130/2003/QĐ-TTg
Không có chunks trong law_id: 130/2003/QĐ-TTg

In [17]:
df[df['titles'].apply(lambda x: not x)]

,id,law_id,aid,content_Article,titles
1175,44,31/NQ-CP,1175,Chính phủ thống nhất đánh giá: Trong bối cảnh\...,[]
1176,44,31/NQ-CP,1176,\t\ta) Khẩn trương hoàn thành việc xây dựng: B...,[]
1177,44,31/NQ-CP,1177,\t\ta) Thực hiện chính sách tài khóa mở rộng h...,[]
1178,44,31/NQ-CP,1178,\t\ta) Điều hành các công cụ chính sách tiền t...,[]
1179,44,31/NQ-CP,1179,"\t\ta) Rà soát, thúc đẩy phát triển và cơ cấu ...",[]
...,...,...,...,...,...
59282,2133,30/2024/NĐ-CP,59282,"Các Bộ trưởng, Thủ trưởng cơ quan ngang bộ, Th...",[]
59578,2150,22/2024/TT-BTC,59578,"1. Sửa đổi, bổ sung điểm 3 Phụ lục số 01/ĐKHN,...",[]
59579,2150,22/2024/TT-BTC,59579,1. Thay thế cụm từ “Giấy CMND/Hộ chiếu số:...c...,[]
59580,2150,22/2024/TT-BTC,59580,"1. Sửa đổi, bổ sung điểm 3 Phụ lục số 06 ban h...",[]


In [18]:
df.to_parquet("data/law_corpus_with_titles.parquet",
    index=False,
    compression="zstd",
    engine="pyarrow"
)